In [88]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from get_hr import get_hr

In [89]:
csv_path = 'sleep/data/pargat_at_temple_com_2026-05-18.csv'
data = pd.read_csv(csv_path)
data['timestamp_ist'] = pd.to_datetime(data['timestamp'], unit='ms', utc=True).dt.tz_convert('Asia/Kolkata')
start = pd.Timestamp('2026-05-19 07:24:43', tz='Asia/Kolkata')
end = pd.Timestamp('2026-05-19 07:24:58', tz='Asia/Kolkata')
data = data[(data['timestamp_ist'] >= start) & (data['timestamp_ist'] < end)].reset_index(drop=True)

In [90]:
t = pd.to_datetime(data['timestamp'], unit='ms', utc=True).dt.tz_convert('Asia/Kolkata')

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=('green', 'accel_x', 'accel_y', 'accel_z'))
# fig.add_trace(go.Scatter(x=t, y=data['hr'], name='hr'), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=data['green'], name='green'), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=data['accel_x'], name='accel_x'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=data['accel_y'], name='accel_y'), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=data['accel_z'], name='accel_z'), row=4, col=1)
fig.update_xaxes(title_text='time (IST)', row=4, col=1)
fig.update_layout(height=600, showlegend=False)
fig.show()

In [91]:
from scipy.signal import butter, sosfiltfilt

SOURCE_FS = 32.0
TARGET_FS = 26.0

SOS_BP_09_40 = butter(4, [0.9, 4.0], btype="bandpass", fs=TARGET_FS, output="sos")
SOS_BP_06_36 = butter(4, [0.6, 3.6], btype="bandpass", fs=TARGET_FS, output="sos")

GRAVITY_CUTOFF_HZ = 0.2
SOS_LP_GRAVITY = butter(2, GRAVITY_CUTOFF_HZ, btype="lowpass", fs=TARGET_FS, output="sos")

HP_CUTOFF_HZ = 0.5
SOS_HP_ACCEL = butter(2, HP_CUTOFF_HZ, btype="highpass", fs=TARGET_FS, output="sos")


def _resample_linear(x: np.ndarray, source_fs: float, target_fs: float) -> np.ndarray:
    """Linear interpolation, matches resample_signal() in cadence_lock_mitigation.c."""
    n_in = len(x)
    duration = (n_in - 1) / source_fs
    n_out = int(duration * target_fs) + 1
    ratio = source_fs / target_fs
    t = np.arange(n_out, dtype=np.float64) * ratio
    idx0 = np.floor(t).astype(int)
    frac = t - idx0
    idx0 = np.clip(idx0, 0, n_in - 1)
    idx1 = np.clip(idx0 + 1, 0, n_in - 1)
    return x[idx0] + frac * (x[idx1] - x[idx0])

def _temporal_emphasis(x: np.ndarray) -> np.ndarray:
    """Linear ramp 0.5 (oldest) -> 1.0 (newest), matches apply_temporal_emphasis()."""
    n = len(x)
    if n < 2:
        return x
    w = 0.5 + 0.5 * np.arange(n) / (n - 1)
    return x * w

In [92]:
green = _resample_linear(data['green'].to_numpy(), source_fs=SOURCE_FS, target_fs=TARGET_FS)
accel_x = _resample_linear(data['accel_x'].to_numpy(), source_fs=SOURCE_FS, target_fs=TARGET_FS)
accel_y = _resample_linear(data['accel_y'].to_numpy(), source_fs=SOURCE_FS, target_fs=TARGET_FS)
accel_z = _resample_linear(data['accel_z'].to_numpy(), source_fs=SOURCE_FS, target_fs=TARGET_FS)


t_ms = data['timestamp'].to_numpy()
n_out = len(green)
t_rs = t_ms[0] + np.arange(n_out) * (1000.0 / TARGET_FS)

ga_raw = pd.DataFrame({
    'timestamp': t_rs,
    'green': green,
    'accel_x': accel_x,
    'accel_y': accel_y,
    'accel_z': accel_z,
    'accel_xy': np.sqrt(accel_x**2 + accel_y**2),
    'accel_xyz': np.sqrt(accel_x**2 + accel_y**2 + accel_z**2),
})

In [93]:
green = ga_raw['green']
accel_x, accel_y, accel_z = ga_raw['accel_x'], ga_raw['accel_y'], ga_raw['accel_z']

sos = SOS_BP_06_36 if (float(np.std(accel_z)) < 350.0) else SOS_BP_09_40 

green = sosfiltfilt(sos, ga_raw['green'])

# accel_x = accel_x - sosfiltfilt(SOS_LP_GRAVITY, accel_x)
accel_x = sosfiltfilt(SOS_HP_ACCEL, accel_x)
accel_x = sosfiltfilt(sos, accel_x)

# accel_y = accel_y - sosfiltfilt(SOS_LP_GRAVITY, accel_y)
accel_y = sosfiltfilt(SOS_HP_ACCEL, accel_y)
accel_y = sosfiltfilt(sos, accel_y)

# accel_z = accel_z - sosfiltfilt(SOS_LP_GRAVITY, accel_z)
# accel_z = sosfiltfilt(SOS_HP_ACCEL, accel_z)
accel_z = sosfiltfilt(sos, accel_z)

ga_filt = pd.DataFrame({
    'timestamp': t_rs,
    'green': green,
    'accel_x': accel_x,
    'accel_y': accel_y,
    'accel_z': accel_z,
    'accel_xy': np.sqrt(accel_x**2 + accel_y**2),
    'accel_xyz': np.sqrt(accel_x**2 + accel_y**2 + accel_z**2),
})

In [94]:
t = pd.to_datetime(ga_raw['timestamp'], unit='ms', utc=True).dt.tz_convert('Asia/Kolkata')

cols = [c for c in ga_raw.columns if c != 'timestamp']
specs = [[{'secondary_y': True}] for _ in cols]
fig = make_subplots(rows=len(cols), cols=1, shared_xaxes=True, subplot_titles=cols, specs=specs)
for i, c in enumerate(cols, start=1):
    # fig.add_trace(
    #     go.Scatter(x=t, y=ga_raw[c], name=f'{c} (raw)',
    #                line=dict(color='red', width=1)),
    #     row=i, col=1, secondary_y=False,
    # )
    fig.add_trace(
        go.Scatter(x=t, y=ga_filt[c], name=f'{c} (filter)',
                   line=dict(color='royalblue', width=1)),
        row=i, col=1, secondary_y=False,
    )
fig.update_xaxes(title_text='time (IST)', row=len(cols), col=1)
fig.update_layout(height=180 * len(cols), showlegend=True)
fig.show()

In [95]:
HR_MIN_BPM = 36.0
HR_MAX_BPM = 220.0
FREQ_MIN_HZ = HR_MIN_BPM / 60.0
FREQ_MAX_HZ = HR_MAX_BPM / 60.0

FFT_SIZE = 1024

def _rfft_magnitude(x: np.ndarray) -> tuple[np.ndarray, float]:
    fft_in = np.zeros(FFT_SIZE, dtype=np.float64)
    n = min(len(x), FFT_SIZE)
    fft_in[:n] = x[:n]
    fft_in[:n] -= fft_in[:n].mean()
    fft_in[:n] *= np.hanning(n)
    # if n >= 2:```
    #     fft_in[:n] *= 0.5 + 0.5 * np.arange(n) / (n - 1)
    mags = np.abs(np.fft.rfft(fft_in))
    freq_res = TARGET_FS / FFT_SIZE
    return mags, freq_res

def _find_dominant_peak(mags: np.ndarray, freq_res: float,
                        f_lo: float, f_hi: float) -> tuple[float, float]:
    """Argmax over [f_lo, f_hi]. Returns (freq, mag)."""
    fft_half = FFT_SIZE // 2 + 1
    k_lo = max(0, int(f_lo / freq_res))
    k_hi = min(fft_half - 1, int(f_hi / freq_res))
    if k_hi < k_lo:
        return 0.0, 0.0
    sub = mags[k_lo:k_hi + 1]
    k = int(np.argmax(sub)) + k_lo
    return k * freq_res, float(mags[k])


In [96]:
green_mags, freq_res = _rfft_magnitude(ga_filt['green'])
green_freq, _ = _find_dominant_peak(green_mags, freq_res, FREQ_MIN_HZ, FREQ_MAX_HZ)

accel_xy_mags, _ = _rfft_magnitude(ga_filt['accel_xy'])
accel_xy_freq, _ = _find_dominant_peak(accel_xy_mags, freq_res, FREQ_MIN_HZ, FREQ_MAX_HZ)

accel_x_mags, _ = _rfft_magnitude(ga_filt['accel_x'])
accel_x_freq, _ = _find_dominant_peak(accel_x_mags, freq_res, FREQ_MIN_HZ, FREQ_MAX_HZ)

accel_y_mags, _ = _rfft_magnitude(ga_filt['accel_y'])
accel_y_freq, _ = _find_dominant_peak(accel_y_mags, freq_res, FREQ_MIN_HZ, FREQ_MAX_HZ)

accel_z_mags, _ = _rfft_magnitude(ga_filt['accel_z'])
accel_z_freq, _ = _find_dominant_peak(accel_z_mags, freq_res, FREQ_MIN_HZ, FREQ_MAX_HZ)

(float(green_freq * 60.0), float(accel_xy_freq * 60.0), float(accel_z_freq * 60.0))


(53.3203125, 54.84375, 70.078125)

In [97]:
import os
from pull_temple_data import get_firmware_hr, get_user_id_by_email

stem = os.path.basename(csv_path).removesuffix('.csv').rsplit('_', 1)[0]
local, domain = stem.split('_at_', 1)
email = f"{local}@{domain.replace('_', '.')}"

user_id = get_user_id_by_email(email)
hr = get_firmware_hr(user_id, start.strftime('%Y-%m-%d %H:%M:%S'), end.strftime('%Y-%m-%d %H:%M:%S'))

In [98]:
fft_half = FFT_SIZE // 2 + 1
freqs = np.arange(fft_half) * freq_res
band = (freqs >= FREQ_MIN_HZ) & (freqs <= FREQ_MAX_HZ)
freqs_bpm = freqs * 60.0

firmware_hr = float(hr['firmware_hr'].mean()) if hr is not None and not hr.empty else 0.0

fig = make_subplots(rows=5, cols=1, shared_xaxes=True,
                    subplot_titles=('green spectrum', 'accel_x spectrum', 'accel_y spectrum', 'accel_z spectrum', 'accel_xy spectrum'))

rows = [
    ('green',    green_mags,    green_freq),
    ('accel_x',  accel_x_mags,  accel_x_freq),
    ('accel_y',  accel_y_mags,  accel_y_freq),
    ('accel_z',  accel_z_mags,  accel_z_freq),
    ('accel_xy', accel_xy_mags, accel_xy_freq),
]

color = 'seagreen'

for row, (name, mags, peak_hz) in enumerate(rows, start=1):
    y_max = float(mags[band].max())
    fig.add_trace(go.Scatter(x=freqs_bpm[band], y=mags[band], line=dict(color=color)),
                  row=row, col=1)
    fig.add_vline(x=peak_hz * 60.0, line_dash='dash', line_color=color, row=row, col=1)
    fig.add_annotation(x=peak_hz * 60.0, y=y_max,
                       text=f'{peak_hz * 60.0:.1f}',
                       font=dict(color=color),
                       showarrow=True, arrowhead=2,
                       ax=60, ay=-25,
                       row=row, col=1)
    # if firmware_hr > 0:
    #     fig.add_vline(x=firmware_hr, line_dash='solid', line_color='black', row=row, col=1)

fig.update_xaxes(title_text='bpm', row=3, col=1)
fig.update_layout(height=700, showlegend=True,
                  title_text=f'Firmware HR: {firmware_hr:.1f} bpm')
fig.show()